In [0]:
catalog_name = "ecommerce_catalog"
schema_name = "ecom_schema"

spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

df_sales_raw = spark.table("raw_sales")
df_customers_raw = spark.table("raw_customers")
df_api_users_raw = spark.table("raw_api_users")

print("Silver Notebook Cleaning")

In [0]:
df_api_users_raw.display()

In [0]:
from pyspark.sql.functions import col, lower, regexp_replace

In [0]:
df_api_silver = df_api_users_raw.select(
    col("id").alias("api_user_id"),
    col("name").alias("full_name"),
    lower(col("email")).alias("email"),
    col("city").alias("city"),
    col("zipcode").alias("zipcode"),
    col("company_name").alias("company_name"),
    col("phone")
)

In [0]:
df_api_silver = df_api_silver.withColumn("full_name", regexp_replace(col("full_name"), "^(Mr\.|Mrs\.|Ms\.|Dr\.) ", ""))

In [0]:
df_api_silver = df_api_silver.withColumn("phone", regexp_replace(col("phone"), "[^0-9]", ""))

In [0]:
df_api_silver.display()

In [0]:
from pyspark.sql.functions import to_timestamp, round, year, month, day, col

In [0]:
df_sales_silver = df_sales_raw.select(
    col("transaction_id"),
    col("customer_id"),
    col("product_id"),
    round(col("amount"), 2).alias("amount"),
    to_timestamp(col("transaction_date")).alias("transaction_timestamp")
)

In [0]:
df_sales_silver = df_sales_silver.withColumn("sales_year", year(col("transaction_timestamp"))) \
                                 .withColumn("sales_month", month(col("transaction_timestamp"))) \
                                 .withColumn("sales_day", day(col("transaction_timestamp")))

In [0]:
df_sales_silver = df_sales_silver.filter(col("amount") > 0)

In [0]:
display(df_sales_silver.limit(5))

In [0]:
df_sales_silver.printSchema()

In [0]:
from pyspark.sql.functions import col, lower, to_date

In [0]:
df_customers_silver = df_customers_raw.select(
    col("customer_id"),
    col("name"),
    lower(col("email")).alias("email"),
    col("country"),
    to_date(col("signup_date")).alias("signup_date")
)

In [0]:
df_customers_silver = df_customers_silver.filter(col("customer_id").isNotNull() & col("email").isNotNull())

In [0]:
df_customers_silver = df_customers_silver.dropDuplicates(["customer_id"])

In [0]:
display(df_customers_silver.limit(5))

In [0]:
df_customer_profile_full = df_customers_silver.join(
    df_api_silver, 
    on="email", 
    how="inner"
    ).select(
    df_customers_silver["customer_id"],
    df_customers_silver["name"],
    df_customers_silver["email"],
    df_customers_silver["country"],
    df_api_silver["city"],
    df_api_silver["phone"],
    df_customers_silver["signup_date"]
    )

In [0]:
display(df_customer_profile_full.limit(5))

In [0]:
from pyspark.sql.functions import *

In [0]:
df_sales_agg = df_sales_silver.groupBy("customer_id").agg(
    sum("amount").alias("total_spent"),
    count("transaction_id").alias("total_transactions")
)

In [0]:
df_full_customers = df_customer_profile_full.join(
    df_sales_agg, 
    on="customer_id", 
    how="left" 
)

In [0]:
df_full_customers = df_full_customers.fillna(0, subset=["total_spent", "total_transactions"])

In [0]:
display(df_full_customers.limit(5))

In [0]:
df_full_customers.display()

In [0]:
df_full_customers.printSchema()

In [0]:
df_api_one = df_api_silver.withColumn("api_user_id", col("api_user_id") + 100)

df_customer_profile_full = df_customers_silver.join(
    df_api_one, 
    df_customers_silver.customer_id == df_api_one.api_user_id, 
    how="inner"
).select(
    df_customers_silver["customer_id"],
    df_customers_silver["name"],
    df_customers_silver["email"],
    df_customers_silver["country"],
    df_api_one["city"],
    df_api_one["phone"],
    df_customers_silver["signup_date"]
)
df_full_data = df_customer_profile_full.join(
    df_sales_agg, 
    on="customer_id", 
    how="left"
).fillna(0)

display(df_full_data)

In [0]:
df_api_silver.write.format("delta").mode("overwrite").saveAsTable("silver_api_users")
df_sales_silver.write.format("delta").mode("overwrite").saveAsTable("silver_sales")
df_customers_silver.write.format("delta").mode("overwrite").saveAsTable("silver_customers")
df_full_data.write.format("delta").mode("overwrite").saveAsTable("silver_full_data")
print("ok")